# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [1]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 💾 Charger les données

In [2]:
import json
import ast
from pathlib import Path

def load_pyomo_data(input_path="../data/Pastissimo_data.json"):
    """Charge les données JSON et convertit les clés string en types natifs."""
    input_file = Path(input_path)
    with open(input_file, "r") as f:
        data = json.load(f)

    def _convert_key(key):
        if not isinstance(key, str):
            return key
        if key.startswith("(") and key.endswith(")"):
            try:
                return ast.literal_eval(key)
            except Exception:
                return key
        try:
            return int(key)
        except Exception:
            return key

    # Convertir les dictionnaires de parametres indexes
    params = data.get("params", {})
    for pname, pval in list(params.items()):
        if isinstance(pval, dict):
            params[pname] = {_convert_key(k): v for k, v in pval.items()}

    cartesian = data.get("cartesian_data", {})
    for cname, cval in list(cartesian.items()):
        if isinstance(cval, dict):
            cartesian[cname] = {_convert_key(k): v for k, v in cval.items()}

    data["params"] = params
    data["cartesian_data"] = cartesian
    return data

# Charger les données
data = load_pyomo_data()

## 🔹 Model

In [3]:
from pyomo.environ import *

model = ConcreteModel()

## 🔹 Sets

In [4]:
model.PERIODES = Set(initialize=data['sets']['PERIODES'])

## 🔹 Parameters

In [5]:
model.Cout_achat = Param(model.PERIODES, initialize=data['params']['Cout_achat'], within=NonNegativeReals)
model.Cout_prod = Param(model.PERIODES, initialize=data['params']['Cout_prod'], within=NonNegativeReals)
model.mini = Param(model.PERIODES, initialize=data['params']['mini'], within=NonNegativeReals)
model.maxi = Param(model.PERIODES, initialize=data['params']['maxi'], within=NonNegativeReals)
model.cap_prod = Param(model.PERIODES, initialize=data['params']['cap_prod'], within=NonNegativeReals)
model.cap_ble = Param(initialize=data['params']['cap_ble'], within=NonNegativeReals)
model.cap_spag = Param(initialize=data['params']['cap_spag'], within=NonNegativeReals)
model.cout_Stock_ble = Param(initialize=data['params']['cout_Stock_ble'], within=NonNegativeReals)
model.cout_Stock_spag = Param(initialize=data['params']['cout_Stock_spag'], within=NonNegativeReals)

## 🔹 Variables

In [6]:
model.Achat_ble = Var(model.PERIODES, domain=NonNegativeReals)
model.Prod_spag = Var(model.PERIODES, domain=NonNegativeReals)
model.Stock_ble = Var(model.PERIODES, domain=NonNegativeReals)
model.Stock_spag = Var(model.PERIODES, domain=NonNegativeReals)

## 🔹 Constraints

In [7]:
model.c0 = Constraint(expr=model.Stock_ble[1] == 2 + model.Achat_ble[1] - model.Prod_spag[1])
model.c1 = Constraint(expr=model.Stock_ble[6] == 2)
model.c2 = Constraint(expr=model.Stock_spag[1] == model.Prod_spag[1] - 4)
model.c3 = Constraint(expr=model.Stock_spag[6] == 0)
model.c_for_0 = ConstraintList()
for p in model.PERIODES:
    model.c_for_0.add(model.Achat_ble[p] >= model.mini[p])
model.c_for_1 = ConstraintList()
for p in model.PERIODES:
    model.c_for_1.add(model.Achat_ble[p] <= model.maxi[p])
model.c_for_2 = ConstraintList()
for p in model.PERIODES:
    model.c_for_2.add(model.Prod_spag[p] <= model.cap_prod[p])
model.c_for_3 = ConstraintList()
for p in model.PERIODES:
    model.c_for_3.add(model.Stock_ble[p] <= model.cap_ble)
model.c_for_4 = ConstraintList()
for p in model.PERIODES:
    model.c_for_4.add(model.Stock_spag[p] <= model.cap_spag)
model.c_for_5 = ConstraintList()
for i in model.PERIODES:
    if i >= 2:
        model.c_for_5.add(model.Stock_ble[i] == model.Stock_ble[i-1] + model.Achat_ble[i] - model.Prod_spag[i])
model.c_for_6 = ConstraintList()
for i in model.PERIODES:
    if i >= 2:
        model.c_for_6.add(model.Stock_spag[i] == model.Stock_spag[i-1] + model.Prod_spag[i] - 4)

## 🔹 Objective

In [8]:
model.obj = Objective(expr=sum(model.Cout_achat[p] * model.Achat_ble[p] + model.Cout_prod[p] * model.Prod_spag[p] + model.cout_Stock_ble * model.Stock_ble[p] + model.cout_Stock_spag * model.Stock_spag[p] for p in model.PERIODES), sense=minimize)

## ⚙️ Résolution du modèle

In [9]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

✅ Solver status: ok
✅ Termination condition: optimal


## 🎯 Valeur de la fonction objective

In [10]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

Objectif: obj
Valeur optimale: 28095.0000
Sens: Minimisation


## 📊 Valeurs optimales des variables

In [11]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')

,Variable,Index,Valeur
0,Achat_ble,1,4.0000
1,Achat_ble,2,3.0000
2,Achat_ble,3,5.0000
3,Achat_ble,4,3.0000
4,Achat_ble,5,4.0000
5,Achat_ble,6,5.0000
6,Prod_spag,1,4.0000
7,Prod_spag,2,5.0000
8,Prod_spag,3,4.0000
9,Prod_spag,4,4.0000
